# LEPTOLIC / INT182 รอบ 3 — แนวทาง 1

อ่าน `round3/REVIEW_TH.md` ก่อนรัน: ผลเดิม AUC 0.6761, Brier 0.05653, Recall 0% ที่ 0.5 เป็น **saved outputs** ยังไม่ได้รันยืนยันใหม่

Notebook นี้คง target เดิม เปรียบเทียบ 4/5 features กับ Logistic/Tree เลือกจาก Tune และคง Platt calibration ไว้ล่วงหน้า
ระดับ 1–3 เป็นช่วงคะแนนสัมพัทธ์จาก Tune ไม่ใช่เกณฑ์คลินิกหรือความเสี่ยงที่ยืนยันกับคนไทย

ต้องรันใน checkout `dev-by-gunner` ที่มีโฟลเดอร์ `round3` พร้อม dependencies
หากใช้ Colab ให้เตรียม checkout และเปลี่ยน working directory มาที่ checkout ก่อน
CDC ต้องเข้าถึงได้; runtime ที่พัฒนา notebook นี้ถูก proxy บล็อก 403 ไม่มีผลโมเดลใหม่แนบมา

In [ ]:
%pip install -r round3/requirements.txt

## ตรวจหลักฐานและ protocol ก่อนฝึก
ไม่ใช้ผล Test เลือกฟีเจอร์/threshold และไม่อ้างว่า Test เดิมเป็นข้อมูลใหม่ที่ไม่เคยเห็น

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
display(Markdown(Path('round3/REVIEW_TH.md').read_text(encoding='utf-8')))

## ฝึกด้วยข้อมูลจริง
ขั้นนี้จะหยุดหากดาวน์โหลดหรือการตรวจข้อมูลล้มเหลว ไม่มีการแทนด้วยข้อมูลจำลอง
เลือก work directory ใหม่หากมีผลรอบก่อนแล้ว

In [ ]:
from round3.workflow import run
WORK = Path('round3_work').resolve()
results = run(WORK)

## อ่านผลและระดับ
ตรวจจำนวนคน/เหตุการณ์และช่วงความไม่แน่นอนของทุกระดับ ระดับสูงอาจมี observed rate ไม่สูงขึ้นใน Test ขนาดเล็ก ต้องรายงานตามจริง

In [ ]:
print(results['test_selected'])
print(results['test_levels'])
print(results['test_uncertainty'])
print(results['limitations'])

## ทดลองกรอกข้อมูล
ตัวอย่างด้านล่างอาจถูกปฏิเสธถ้าอยู่นอก training range; ไม่เหมาะกับผู้เคยได้รับการวินิจฉัยแล้ว
ไม่ใช่เครื่องมือวินิจฉัย/เลือกการรักษา

In [ ]:
from round3.predict import predict_file
INPUT = dict(age_years=50, sex_female=0, baseline_sbp=120, baseline_dbp=75, sleep_hours=7)
answer = predict_file(WORK/'round3_outputs'/'model.json', INPUT, previously_diagnosed=False)
print(answer)

## ส่งออก
ไฟล์ ZIP อยู่ใน work directory หลังฝึกสำเร็จ ห้ามแทนที่ผลจริงด้วย test fixtures

In [ ]:
from IPython.display import FileLink
display(FileLink(str(WORK/'LEPTOLIC_round3_outputs.zip')))